# Assumption checking short use cases

## Pre-modeling assumptions

In [3]:
import pandas as pd

dataset_path = "../data/all_data/abortion_bf15.csv"
df = pd.read_csv(dataset_path)
df.head()

,fip,age,race,year,sex,totcase,totpop,rate,totrate,id,...,female,lnr,t,younger,fa,pi,wm15,wf15,bm15,bf15
0,1.0,15.0,2.0,1985.0,2,5683.0,106187,6527.5,5351.9,14.0,...,1.0,8.783779,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
1,1.0,15.0,2.0,1986.0,2,5344.0,106831,6351.2,5002.3,14.0,...,1.0,8.756399,2.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
2,1.0,15.0,2.0,1987.0,2,4983.0,106496,5759.1,4679.0,14.0,...,1.0,8.658537,3.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0
3,1.0,15.0,2.0,1988.0,2,5276.0,105238,6139.6,5013.4,14.0,...,1.0,8.722515,4.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0
4,1.0,15.0,2.0,1989.0,2,5692.0,102956,5951.5,5528.6,14.0,...,1.0,8.691399,5.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0


In [4]:
from cais.methods.pre_model_assumption_utils import check_cond_ignorability

In [5]:
covariates = ['crack', 'alcohol', 'income', 'ur', 'poverty', 'black', 'perc1519']
result_cond_ignorability = check_cond_ignorability(df, 'repeal', covariates)
print(result_cond_ignorability)

{'passed': False, 'reasoning': "Randomization check on 7 covariates (|SMD| < 0.1). Imbalanced: ['crack', 'alcohol', 'income', 'ur', 'poverty', 'black', 'perc1519'].", 'details': {'smds': {'crack': np.float64(0.1901831312076417), 'alcohol': np.float64(0.3090489010094973), 'income': np.float64(0.5807936072744352), 'ur': np.float64(0.4436246174561076), 'poverty': np.float64(-0.2878371568806569), 'black': np.float64(-0.630179818114365), 'perc1519': np.float64(-0.5894732981847037)}, 'threshold': 0.1, 'imbalanced': {'crack': np.float64(0.1901831312076417), 'alcohol': np.float64(0.3090489010094973), 'income': np.float64(0.5807936072744352), 'ur': np.float64(0.4436246174561076), 'poverty': np.float64(-0.2878371568806569), 'black': np.float64(-0.630179818114365), 'perc1519': np.float64(-0.5894732981847037)}}}


In [6]:
from cais.methods.pre_model_assumption_utils import check_positivity
from cais.methods.propensity_score.base import estimate_propensity_scores

In [7]:
ps = estimate_propensity_scores(df, 'repeal', covariates)

overlap_result = check_positivity(df, 'repeal', ps)
print(overlap_result)

{'passed': False, 'reasoning': 'Overlap proportion: 0.777 (threshold 0.5). 472 obs (64.0%) outside [0.1, 0.9]. Consider trimming or restricting to common support.', 'details': {'treated_range': (0.01873818828478758, 0.766475780981912), 'control_range': (0.01, 0.6066358199069207), 'overlap_range': (0.01873818828478758, 0.6066358199069207), 'overlap_proportion': 0.7771532762873611, 'sufficient_overlap': np.True_, 'n_extreme_ps': 472, 'pct_extreme_ps': 0.6404341926729986}}


### Fix an OPENAI_API_KEY in environment before running cells below

In [8]:
from cais.config import get_llm_client
llm = get_llm_client()

In [9]:
from cais.methods.pre_model_assumption_utils import check_sutva

dataset_description = (
    "Panel data of 51 US states from 1985-2000 (Donohue & Levitt). "
    "Treatment 'repeal' is binary: whether the state legalized abortion "
    "before Roe v. Wade in 1973. Outcome 'rate' is the crime rate per 100k. "
    "The hypothesis is that legalized abortion reduced unwanted births, "
    "which later reduced crime. Subgroup: black females age 15."
)

variables_info = {
    'treatment': 'repeal',
    'outcome': 'rate',
    'covariates': covariates,
    'panel_id': 'fip (state FIPS code)',
    'time': 'year (1985-2000)',
}

result_sutva = check_sutva(dataset_description, variables_info, llm=llm)
print(result_sutva)

{'passed': False, 'reasoning': 'The assumption of no interference is likely violated due to potential spillover effects among states. For instance, the legalization of abortion in one state could influence neighboring states through migration patterns, shared economic conditions, or social norms, thereby affecting their crime rates. Additionally, the treatment is binary and may not account for variations in how states implemented the repeal, suggesting the presence of hidden versions of the treatment.', 'details': {'assumption': 'SUTVA (Stable Unit Treatment Value Assumption)'}}


In [10]:
from cais.methods.pre_model_assumption_utils import (
    check_iv_relevance,
    check_iv_exclusion,
    check_iv_exogeneity,
    check_iv_monotonicity,
)

In [11]:
import pandas as pd

In [12]:
dataset_iv_path = "../data/all_data/card_geographic.csv"
df_iv = pd.read_csv(dataset_iv_path)

In [13]:
df_iv = df_iv.drop(columns=['Unnamed: 0'], errors='ignore')
df_iv = df_iv.dropna()

In [14]:
df_iv.head()

,nearc4,educ,black,smsa,south,married,exper,lwage
0,0,7,1,1,0,1.0,16,6.306275
1,0,12,0,1,0,1.0,9,6.175867
2,0,12,0,1,0,1.0,16,6.580639
3,1,11,0,1,0,1.0,10,5.521461
4,1,12,0,1,0,1.0,16,6.591674


In [15]:
# --- check_iv_relevance : Is nearc4 a strong instrument for educ ? ---
result_iv_relevance = check_iv_relevance(
    df=df_iv,
    treatment='educ',
    instruments=['nearc4'],
    covariates=['black', 'smsa', 'south', 'married', 'exper'],
)
print(result_iv_relevance)

{'passed': True, 'reasoning': 'First-stage F = 15.77 (threshold 10.0). Strong instrument.', 'details': {'f_statistic': 15.766661138654587, 'p_value': 7.333887270678314e-05, 'threshold': 10.0}}


In [16]:
# --- check_iv_exclusion ---
card_description = (
    "Card (1995) dataset. 3010 men from the NLS Young Men Cohort. "
    "Treatment is 'educ' (years of education). Instrument is 'nearc4' "
    "(grew up near a 4-year college). Outcome is 'lwage' (log wage). "
    "The exclusion restriction argument is that college proximity affects "
    "education but not wages directly — though this is debated, since "
    "proximity may correlate with local labor market conditions."
)

card_variables = {
    'treatment': 'educ',
    'outcome': 'lwage',
    'instrument': 'nearc4',
    'covariates': ['black', 'smsa', 'south', 'married', 'exper'],
}

result_iv_exclusion = check_iv_exclusion(card_description, card_variables, llm=llm)
print("check_iv_exclusion :", result_iv_exclusion)

check_iv_exclusion : {'passed': False, 'reasoning': 'The exclusion restriction is likely violated because growing up near a 4-year college (nearc4) may influence local labor market conditions, which could directly affect wages (lwage) independent of education (educ). This correlation suggests that the instrument may have a direct effect on the outcome, undermining the validity of the exclusion restriction.', 'details': {'assumption': 'Exclusion restriction'}}


In [17]:
# --- check_iv_exogeneity ---
result_iv_exogeneity = check_iv_exogeneity(card_description, card_variables, llm=llm)
print("check_iv_exogeneity :", result_iv_exogeneity)

check_iv_exogeneity : {'passed': False, 'reasoning': "The assumption of instrument exogeneity is likely violated because 'nearc4' (proximity to a 4-year college) may correlate with local labor market conditions, which can affect wages ('lwage') directly. This suggests that 'nearc4' is not as good as randomly assigned with respect to unobserved confounders affecting both education and wages.", 'details': {'assumption': 'Instrument exogeneity (independence)'}}


In [18]:
# --- check_iv_monotonicity ---
result_iv_monotonicity = check_iv_monotonicity(card_description, card_variables, llm=llm)
print("check_iv_monotonicity :", result_iv_monotonicity)

check_iv_monotonicity : {'passed': None, 'reasoning': "The dataset description does not provide sufficient information about the relationship between the instrument 'nearc4' and the treatment 'educ' to assess the presence of defiers. Without knowing how individuals who grew up near a 4-year college respond to the instrument in terms of their education, we cannot determine if there are units that would decrease their education despite being near a college.", 'details': {'assumption': 'Monotonicity (LATE)'}}


In the description furnished before, there wasn't enough description for the LLM to judge whether IV monocity was valid. That's why it returns `'None'`.

We enrich the description now:

In [19]:
card_description = (
    "Card (1995) dataset. 3010 men from the NLS Young Men Cohort. "
    "Treatment is 'educ' (years of education). Instrument is 'nearc4' "
    "(grew up near a 4-year college). Outcome is 'lwage' (log wage). "
    "The instrument works through reduced cost of attending college: "
    "individuals near a college face lower transportation and housing costs, "
    "making them more likely to attend. It is implausible that proximity "
    "to a college would cause someone to get LESS education — the effect "
    "should go in one direction only (more proximity → more education or no change)."
)

In [20]:
# --- check_iv_monotonicity ---
result_iv_monotonicity = check_iv_monotonicity(card_description, card_variables, llm=llm)
print("check_iv_monotonicity :", result_iv_monotonicity)

check_iv_monotonicity : {'passed': True, 'reasoning': "The assumption of monotonicity is plausibly satisfied because the instrument 'nearc4' is expected to increase the likelihood of attending college due to reduced costs associated with proximity to a college. It is highly unlikely that being near a college would lead to a decrease in education, as the natural effect of proximity is to either increase education or have no effect, thus supporting the absence of defiers.", 'details': {'assumption': 'Monotonicity (LATE)'}}


In [21]:
from cais.methods.pre_model_assumption_utils import (
    check_parallel_trends,
    check_no_anticipation,
    check_baseline_outcome_balance,
    check_stable_group_composition,
)

In [22]:
df_did = pd.read_csv('../data/all_data/castle.csv')
df_did['ever_treated'] = df_did.groupby('sid')['post'].transform('max').astype(int)

# 2006 is the cutoff year for the tests
treatment_year = 2006

df_did.head()

,state,year,sid,cdl,pre2_cdl,caselaw,anywhere,assumption,civil,homicide_c,...,_Iyear_2003,_Iyear_2004,_Iyear_2005,_Iyear_2006,_Iyear_2007,_Iyear_2008,_Iyear_2009,_Iyear_2010,popwt,ever_treated
0,Alabama,2000,1,0.0,0.0,0.0,0,0,0,329,...,0,0,0,0,0,0,0,0,4499293.0,1
1,Alabama,2001,1,0.0,0.0,0.0,0,0,0,379,...,0,0,0,0,0,0,0,0,4499293.0,1
2,Alabama,2002,1,0.0,0.0,0.0,0,0,0,303,...,0,0,0,0,0,0,0,0,4499293.0,1
3,Alabama,2003,1,0.0,0.0,0.0,0,0,0,299,...,1,0,0,0,0,0,0,0,4499293.0,1
4,Alabama,2004,1,0.0,1.0,0.0,0,0,0,254,...,0,1,0,0,0,0,0,0,4499293.0,1


In [23]:
# --- check_parallel_trends ---
result_parallel_trends = check_parallel_trends(
    df=df_did,
    time_var='year',
    outcome='l_homicide',
    group_indicator_col='ever_treated',
    treatment_period_start=treatment_year,
)
print(result_parallel_trends)

{'passed': np.True_, 'reasoning': 'Simple linear trend test: p-value for group-trend interaction: 0.8264. Parallel trends: True.', 'details': {'p_value': np.float64(0.8263558702861501), 'error': None}}


In [24]:
# --- check_no_anticipation ---
covariates = ['l_police', 'l_income', 'l_prisoner', 'unemployrt', 'poverty']

result_no_anticipation = check_no_anticipation(
    df=df_did,
    time_var='year',
    group_var='sid',
    outcome='l_homicide',
    treated_unit_indicator='ever_treated',
    covariates=covariates,
    treatment_period_start=treatment_year,
    placebo_period_start=2003,
)
print(result_no_anticipation)

{'passed': True, 'reasoning': 'Placebo treatment effect estimated at 0.0976 (p=0.1673). Test passed: True.', 'details': {'passed': True, 'effect_estimate': 0.09764229987805968, 'p_value': 0.16731301316128822, 'error': None}}


In [25]:
# --- check_baseline_outcome_balance ---
result_baseline_outcome_balance = check_baseline_outcome_balance(
    df=df_did,
    treatment='ever_treated',
    outcome='l_homicide',
    time_var='year',
    treatment_period_start=treatment_year,
)
print(result_baseline_outcome_balance)

{'passed': np.False_, 'reasoning': 'Baseline outcome SMD = 0.675 (threshold 0.1).', 'details': {'smd_pre_outcome': np.float64(0.6753481392211624), 'threshold': 0.1}}


In [26]:
# --- check_stable_group_composition (LLM-reasoned) ---
castle_description = (
    "Castle Doctrine dataset. Panel of 50 US states from 2000-2010. "
    "Treatment is adoption of Castle Doctrine / Stand Your Ground laws, "
    "which expanded the right to use lethal force in self-defense. "
    "States adopted the law at different times (staggered treatment). "
    "Outcome is log homicide rate. Unit of observation is state-year."
)

castle_variables = {
    'treatment': 'ever_treated',
    'outcome': 'l_homicide',
    'panel_id': 'sid',
    'time': 'year (2000-2010)',
    'covariates': covariates,
}

result_stable_group_composition = check_stable_group_composition(castle_description, castle_variables, llm=llm)
print(result_stable_group_composition)

{'passed': None, 'reasoning': 'The dataset description does not provide information on whether there was differential attrition or selective entry/exit of states in relation to the treatment (adoption of Castle Doctrine laws). Without data on how states may have changed their composition over time or how the treatment affected state characteristics, it is not possible to assess the plausibility of the stable group composition assumption.', 'details': {'assumption': 'Stable group composition'}}


In [27]:
castle_description = (
    "Castle Doctrine dataset. Panel of 50 US states from 2000-2010. "
    "Treatment is adoption of Castle Doctrine / Stand Your Ground laws. "
    "Outcome is log homicide rate. Unit of observation is state-year. "
    "All 50 states are observed for all 11 years — the panel is balanced "
    "with no missing state-year observations. States do not enter or exit "
    "the dataset. However, population migration between states could change "
    "the demographic composition of treatment and control states over time."
)

result_stable_group_composition2 = check_stable_group_composition(castle_description, castle_variables, llm=llm)
print(result_stable_group_composition2)

{'passed': False, 'reasoning': 'While the dataset is balanced and does not have missing observations, the potential for population migration between states introduces the possibility of changes in the demographic composition of treatment and control groups over time. This could lead to differential attrition or selective entry/exit, violating the stable group composition assumption.', 'details': {'assumption': 'Stable group composition'}}


In [28]:
# --- check_sutva (LLM-reasoned) ---
result_sutva_did = check_sutva(castle_description, castle_variables, llm=llm)
print(result_sutva_did)

{'passed': False, 'reasoning': 'The assumption of no interference is likely violated due to potential population migration between states, which could lead to spillover effects where the treatment in one state affects the outcomes in another state. Additionally, the treatment (adoption of Castle Doctrine laws) may not be uniformly applied across states, suggesting the possibility of hidden versions of the treatment. Therefore, SUTVA is not plausibly satisfied.', 'details': {'assumption': 'SUTVA (Stable Unit Treatment Value Assumption)'}}


## Post-modeling assumptions

In [29]:
from cais.methods.post_model_assumption_utils import (
    check_iv_overidentification
)

In [30]:
result_iv_overidentification = check_iv_overidentification(
    sm_results=None,
    df=df_iv,
    treatment='educ',
    outcome='lwage',
    instruments=['nearc4'],
    covariates=['black', 'smsa', 'south', 'married', 'exper'],
)
print(result_iv_overidentification)

{'passed': None, 'reasoning': 'Test not applicable (Need more instruments than endogenous regressors)', 'details': {}}


In [31]:
from statsmodels.sandbox.regression.gmm import IV2SLS
import statsmodels.api as sm

# using 'nearc4' and 'smsa' as instruments (smsa isn't a real instrument, just used for testing)
instruments = ['nearc4', 'smsa']
covariates_iv = ['black', 'south', 'married', 'exper']

# IV estimation via 2SLS
endog = df_iv['lwage']
exog = sm.add_constant(df_iv[['educ'] + covariates_iv])
instrument_matrix = sm.add_constant(df_iv[instruments + covariates_iv])

iv_model = IV2SLS(endog, exog, instrument_matrix).fit()

result_iv_overidentification2 = check_iv_overidentification(
    sm_results=iv_model,
    df=df_iv,
    treatment='educ',
    outcome='lwage',
    instruments=instruments,
    covariates=covariates_iv,
)
print(result_iv_overidentification2)

{'passed': np.False_, 'reasoning': 'Sargan-Hansen test: statistic=8.72, p=0.0031. Instruments may be invalid — correlated with errors.', 'details': {'statistic': np.float64(8.724238008268228), 'p_value': np.float64(0.0031400727800432876), 'status': 'Test successful'}}
